# 从全连接层到卷积

> 学习目标：理解 CNN 为什么适合处理图像，以及卷积层如何从全连接层推导出来。

## 1. 为什么需要卷积神经网络？

多层感知机（MLP）将图像展平成向量，忽略了像素的**空间结构**，而且参数量很大。

例如，输入为 $10^6$ 个像素、隐藏层为 $10^3$ 个神经元时，仅一层的权重参数量就是：

$$
10^6\times10^3=10^9
$$

即 **10 亿个权重**（还未计入偏置）。CNN 利用图像的局部结构，通过**局部连接**和**权重共享**显著减少参数。

## 2. 卷积层的两个关键思想

- **平移等变性（translation equivariance）**：输入中的目标移动，卷积产生的特征响应也随之移动；同一卷积核可以在不同位置识别相同特征。原文将这一动机称作“平移不变性”，但严格来说，卷积层本身主要具有平移等变性。
- **局部性（locality）**：每次只处理图像的一小块区域，逐层组合局部特征，最终得到更大范围的信息。

### 插图：寻找沃尔多

![沃尔多游戏示例图](../img/where-wally-walker-books.jpg)

> 同一个“沃尔多检测器”可以扫描整张图片，不需要为不同位置设计不同的检测参数。

## 3. 从全连接层推导到卷积层

### 3.1 全连接层的数学形式

设二维输入图像为 $\mathbf X$，隐藏表示为 $\mathbf H$，权重为四阶张量 $\mathsf W$，偏置为 $\mathbf U$：

$$
[\mathbf H]_{i,j}=[\mathbf U]_{i,j}+\sum_k\sum_l[\mathsf W]_{i,j,k,l}[\mathbf X]_{k,l}
$$

**符号说明：**
- $(i,j)$：输出位置；$(k,l)$：输入位置。
- $[\mathsf W]_{i,j,k,l}$：输入位置 $(k,l)$ 对输出位置 $(i,j)$ 的连接权重。

令 $k=i+a$、$l=j+b$，把输入坐标改写为相对输出位置的**偏移量**：

$$
[\mathbf H]_{i,j}=[\mathbf U]_{i,j}+\sum_a\sum_b[\mathsf V]_{i,j,a,b}[\mathbf X]_{i+a,j+b}
$$

其中

$$
[\mathsf V]_{i,j,a,b}=[\mathsf W]_{i,j,i+a,j+b}
$$

> 这一步只是**重新编号**，没有减少参数。

### 3.2 权重共享：不再依赖输出位置

为了让相同特征在不同位置使用相同的检测方法，设

$$
[\mathsf V]_{i,j,a,b}=[\mathbf V]_{a,b},\qquad [\mathbf U]_{i,j}=u
$$

则

$$
[\mathbf H]_{i,j}=u+\sum_a\sum_b[\mathbf V]_{a,b}[\mathbf X]_{i+a,j+b}
$$

**理解：**权重只取决于局部偏移 $(a,b)$，不取决于卷积核正在扫描的位置 $(i,j)$。这就是**权重共享**。

### 3.3 局部连接：限制感受范围

只考虑距离当前位置不超过 $\Delta$ 的像素：

$$
\boxed{
[\mathbf H]_{i,j}=u+\sum_{a=-\Delta}^{\Delta}\sum_{b=-\Delta}^{\Delta}[\mathbf V]_{a,b}[\mathbf X]_{i+a,j+b}
}
$$

- $\mathbf V$：卷积核（kernel）或滤波器（filter）。
- $\Delta$：邻域半径；卷积核大小为 $(2\Delta+1)\times(2\Delta+1)$。
- 例如 $\Delta=1$ 对应 $3\times3$ 卷积核，仅有 **9 个核权重 + 1 个偏置**（单输入、单输出通道）。

> **结论：权重共享 + 局部连接 = CNN 参数高效的关键。**

## 4. 数学卷积与深度学习中的互相关

连续卷积：

$$
(f*g)(\mathbf x)=\int f(\mathbf z)g(\mathbf x-\mathbf z)\,d\mathbf z
$$

一维离散卷积：

$$
(f*g)(i)=\sum_a f(a)g(i-a)
$$

二维离散卷积：

$$
(f*g)(i,j)=\sum_a\sum_b f(a,b)g(i-a,j-b)
$$

而深度学习卷积层通常执行**互相关（cross-correlation）**：

$$
H_{i,j}=\sum_a\sum_b V_{a,b}X_{i+a,j+b}
$$

两者的关键区别是**卷积需要翻转核，互相关不翻转核**。因为核参数可学习，实际训练时通常仍将互相关操作称为“卷积”。

## 5. 卷积核如何寻找目标？

![卷积核扫描图片，寻找沃尔多](../img/waldo-mask.jpg)

卷积核在图像上逐位置滑动，对每个局部窗口加权计算，形成一张**特征图（feature map）**。其中响应较大的位置可能包含所寻找的特征。

## 6. 多输入与多输出通道

彩色图像含三个输入通道：RGB。设输入 $\mathsf X$ 的形状为 $H\times W\times C_{\mathrm{in}}$，输出 $\mathsf H$ 有 $C_{\mathrm{out}}$ 个通道。

多通道卷积为：

$$
\boxed{
[\mathsf H]_{i,j,d}=\sum_{a=-\Delta}^{\Delta}\sum_{b=-\Delta}^{\Delta}\sum_{c=1}^{C_{\mathrm{in}}}[\mathsf V]_{a,b,c,d}[\mathsf X]_{i+a,j+b,c}+u_d
}
$$

**索引解释：**
- $(i,j)$：输出特征图的空间位置。
- $(a,b)$：卷积核的空间偏移。
- $c$：输入通道，例如 R、G、B。
- $d$：输出通道，每个输出通道对应一组独立卷积核及偏置 $u_d$。

例如，一个 $3\times3$ 卷积层包含 **3 个输入通道、16 个输出通道**，参数量为：

$$
3\times3\times3\times16+16=448
$$

其中 $432$ 是权重、$16$ 是偏置。一般的参数量公式为：

$$
\boxed{K_hK_wC_{\mathrm{in}}C_{\mathrm{out}}+C_{\mathrm{out}}}
$$

> 上式假设每个输出通道都有一个偏置，且使用普通卷积而非分组卷积。实际输出空间大小还受填充（padding）和步幅（stride）影响。

## 7. 总结

| 概念 | 作用 |
| --- | --- |
| 局部连接 | 只观察附近像素，减少连接数 |
| 权重共享 | 同一个卷积核在不同位置重复使用 |
| 平移等变性 | 目标平移时，特征响应相应平移 |
| 卷积核 | 在局部区域提取边缘、纹理等特征 |
| 多通道 | 融合不同输入特征，并生成多种输出特征图 |

**一句话记忆：**CNN 用少量共享的卷积核扫描图像，提取局部特征，再逐层组合成更复杂的表示。

---

**插图使用说明：**本文保留了原教材的相对图片路径 `../img/...`。若单独移动本 Markdown 文件，需要将两张图片同步放入相对于 Markdown 文件的 `../img/` 目录，否则 GitHub 可能无法显示。
